# 08 - pAgo QC evidence inventory

This notebook calls the pAgo QC evidence inventory snapshot module. The evidence rules live in `src/pago_pipeline/pago_qc.py`; artifact writing lives in `src/pago_pipeline/pago_qc_snapshot.py`.

In [1]:
# =============================================================================
# CELL 1 - Imports
# =============================================================================

from __future__ import annotations

import importlib
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

import src.pago_pipeline.ncbi_snapshot as ncbi_snapshot_module
import src.pago_pipeline.pago_qc_snapshot as pago_qc_snapshot_module

# Reload pipeline modules so notebook reruns pick up local code changes.
ncbi_snapshot_module = importlib.reload(ncbi_snapshot_module)
pago_qc_snapshot_module = importlib.reload(pago_qc_snapshot_module)

SnapshotMode = ncbi_snapshot_module.SnapshotMode
resolve_pago_qc_evidence_inventory = (
    pago_qc_snapshot_module.resolve_pago_qc_evidence_inventory
)

In [2]:
# =============================================================================
# CELL 2 - Load environment and resolve project root
# =============================================================================

dotenv_path = find_dotenv(usecwd=False)

if not dotenv_path:
    raise FileNotFoundError(
        "Could not find a .env file while walking up parent directories. "
        "Place .env with your project configuration at the project root."
    )

load_dotenv(dotenv_path=dotenv_path, override=True)

PROJECT_ROOT = Path(dotenv_path).resolve().parent

print(f"Project root: {PROJECT_ROOT}")

Project root: C:\Programming\Python\pAgo-project


In [3]:
# =============================================================================
# CELL 3 - Define input and output paths
# =============================================================================

METADATA_CSV_PATH = (
    PROJECT_ROOT
    / "data"
    / "02-intermediate"
    / "protein_metadata_csv"
    / "latest"
    / "protein_metadata.csv"
)

SNAPSHOT_ROOT_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "03-features"
    / "pago_qc"
    / "evidence_inventory"
)

print(f"Metadata CSV: {METADATA_CSV_PATH}")
print(f"Snapshot root directory: {SNAPSHOT_ROOT_DIRECTORY}")

Metadata CSV: C:\Programming\Python\pAgo-project\data\02-intermediate\protein_metadata_csv\latest\protein_metadata.csv
Snapshot root directory: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory


In [4]:
# =============================================================================
# CELL 4 - Run evidence inventory
# =============================================================================

inventory_payload = resolve_pago_qc_evidence_inventory(
    snapshot_mode=SnapshotMode.reuse_latest_or_create,
    metadata_csv_file_path=METADATA_CSV_PATH,
    snapshot_root_directory=SNAPSHOT_ROOT_DIRECTORY,
)

inventory_manifest = inventory_payload["manifest"]

print(f"Metadata rows: {inventory_manifest['metadata_row_count']:,}")
print(f"Snapshot directory: {inventory_payload['snapshot_directory']}")
print(f"Evidence flags: {inventory_payload['evidence_flags_file_path']}")
print(f"Evidence counts: {inventory_payload['evidence_counts_file_path']}")
print(f"Manifest: {inventory_payload['manifest_file_path']}")

Latest pAgo QC evidence inventory snapshot is available. Reusing frozen snapshot.
Metadata rows: 41,345
Snapshot directory: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\latest
Evidence flags: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\latest\evidence_flags.csv
Evidence counts: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\latest\evidence_counts.csv
Manifest: C:\Programming\Python\pAgo-project\data\03-features\pago_qc\evidence_inventory\latest\manifest.json


In [5]:
# =============================================================================
# CELL 5 - Inspect first-pass evidence counts
# =============================================================================

evidence_counts_df = inventory_payload["evidence_counts"]
evidence_counts_df.sort_values("count", ascending=False)

,flag,count,fraction
8,has_cdd_region,39105,0.945822
1,has_piwi_region,24229,0.586020
14,is_short_lt_300,14373,0.347636
3,has_ppiwi_re_region,14251,0.344685
15,is_possible_partial_300_599,13622,0.329472
10,has_sam_methyltransferase_term,11883,0.287411
11,is_probable_methyltransferase_noise,11824,0.285984
9,has_ubig_term,11296,0.273213
0,has_piwi_text_anywhere,10130,0.245011
2,has_classic_piwi_region,9978,0.241335
